# 01 - Train Model
Trains 3 models, picks the best one and saves it to `models/`. Run cells top to bottom (Shift + Enter).

## 1. Install libraries (run once)

In [ ]:
%pip install -q scikit-learn pandas numpy joblib

## 2. Imports

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, roc_auc_score

## 3. Load data

In [ ]:
# Input columns and the column we want to predict
FEATURES = ["Has_Patches", "Has_MS", "Owner_Annual_Income_Lakhs"]
TARGET = "Requires_Loan"

# Sample dataset (replace with pd.read_csv("your_file.csv") for real data)
df = pd.DataFrame({
    "Has_Patches": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    "Has_MS": [0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1],
    "Owner_Annual_Income_Lakhs": [4, 12, 3, 18, 5, 15, 4, 22, 6, 9, 3.5, 14, 5.5, 20, 4.5],
    "Requires_Loan": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
})

# Quick checks: size, class balance, missing values
print("Shape:", df.shape)
print(df[TARGET].value_counts())
print("Missing values:", df.isna().sum().sum())

df

## 4. Split into train and test

In [ ]:
X = df[FEATURES]   # inputs
y = df[TARGET]     # answers

# 80% train / 20% test, keeping the same class ratio in both
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 5. Define the models

In [ ]:
RANDOM_STATE = 42

# Each model = (pipeline, settings to try)
candidates = {
    "Logistic Regression": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
    ),
    "Decision Tree": (
        Pipeline([("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
        {"clf__max_depth": [2, 3, 5, None], "clf__min_samples_leaf": [1, 2, 3]},
    ),
    "Random Forest": (
        Pipeline([("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [2, 4, None],
         "clf__min_samples_leaf": [1, 2]},
    ),
}
print("Models:", list(candidates))

## 6. Train and compare (takes 10-30 seconds)

In [ ]:
# Cross-validation setup (folds can't exceed the smallest class size)
n_splits = int(min(5, y_train.value_counts().min()))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

results, fitted = {}, {}

for name, (pipe, grid) in candidates.items():
    # Try every setting in the grid and keep the best one
    search = GridSearchCV(pipe, grid, cv=cv,
                          scoring={"f1": "f1", "logloss": "neg_log_loss"},
                          refit="logloss", n_jobs=-1)
    search.fit(X_train, y_train)
    best = search.best_estimator_

    # Score the best version with several metrics
    cvres = cross_validate(best, X_train, y_train, cv=cv,
                           scoring=["accuracy", "precision", "recall", "f1", "neg_log_loss"])

    # Check on the held-out test data
    proba = best.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba) if y_test.nunique() > 1 else np.nan

    results[name] = {
        "cv_accuracy": round(np.mean(cvres["test_accuracy"]), 4),
        "cv_precision": round(np.mean(cvres["test_precision"]), 4),
        "cv_recall": round(np.mean(cvres["test_recall"]), 4),
        "cv_f1": round(np.mean(cvres["test_f1"]), 4),
        "cv_log_loss": round(-np.mean(cvres["test_neg_log_loss"]), 4),
        "test_roc_auc": None if np.isnan(auc) else round(auc, 4),
    }
    fitted[name] = best
    print("Done:", name)

# Comparison table
pd.DataFrame(results).T

## 7. Pick the best model

In [ ]:
# Highest F1 wins; ties go to lowest log-loss, then the simplest model
preference = ["Logistic Regression", "Decision Tree", "Random Forest"]
best_name = max(results, key=lambda n: (results[n]["cv_f1"],
                                        -results[n]["cv_log_loss"],
                                        -preference.index(n)))
best_model = fitted[best_name]

print("Best model:", best_name)
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))

## 8. Save the model

In [ ]:
# Retrain the winner on all data
best_model.fit(X, y)

# Save the model and its info
Path("models").mkdir(exist_ok=True)
joblib.dump(best_model, "models/loan_model.joblib")

metadata = {
    "best_model": best_name,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "n_samples": int(len(df)),
    "features": FEATURES,
}
Path("models/metadata.json").write_text(json.dumps(metadata, indent=2))

print("Saved files:", [str(f) for f in sorted(Path("models").iterdir())])